# Composite urban index: advisor's method (A) vs corrected method (B)

This notebook builds both indices for the **96 São Paulo districts and 77 Chicago Community Areas (173 units)** and runs the pre-registered tests that compare them. It calls the same functions as `analysis/scripts/build_simple_index_ab.py`, which produced the [methodology report](results/SP_CHI/simple_index_ab_2026_10_01/README.md); the last section checks that this notebook reproduces the published tables exactly.

**Rules of evidence used throughout**

- Every number shown is computed here from repository data. Nothing is typed in by hand.
- The definitions, tests and pass/fail thresholds were fixed in protocol **P-AB-1** ([docs/SIMPLE_INDEX.md](../docs/SIMPLE_INDEX.md)) *before* any A or B value existed, so the thresholds could not be tuned to the result.
- Construction checks stop the run if any input is lost, double-counted or misassigned.

**Run time:** about 2 minutes; peak memory about 1.7 GB. Run the cells in order from the repository's `.venv` kernel.

## 0. Setup

The functions live in `analysis/scripts/build_simple_index_ab.py`. Importing them (instead of copying code) guarantees that the notebook and the published report use one definition of each factor and test.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis/scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis/scripts'))
import build_simple_index_ab as ab

pd.set_option('display.width', 200, 'display.max_columns', 30)
sp = gpd.read_parquet(ROOT / 'analysis/work/prepared/SP/sp_prep_2026_09_10_v3/N02/districts.parquet')
chi = gpd.read_parquet(ROOT / 'analysis/work/prepared/Chicago/chi_local_2026_09_16_v1/districts.parquet')
NAMES = {**{'SP:' + i: n for i, n in zip(sp.district_id, sp.nm_distrito_municipal)}, **dict(zip(chi.unit_id, chi.district_name))}
named = lambda df: df.assign(name=df.index.map(NAMES))

## 1. Units and land area

**Units.** São Paulo municipal districts (`SP:01`–`SP:96`, Brás = `SP:10`) and Chicago Community Areas (`CHI:01`–`CHI:77`).

**Land area** = unit area minus municipal hydrography (rivers, reservoirs, Lake Michigan). It is used only by method B, as the denominator of the count factors.

Why land area matters: the units differ in size by two orders of magnitude, so a raw count partly measures *how big the unit is*.

In [ ]:
U = ab.units()
pd.DataFrame({c: u.land_km2.describe()[['count', 'min', '50%', 'max']] for c, u in U.items()}).round(3)

## 2. The six factors

Each factor is built once per unit as **additive components** (counts, sums and land), so that any group of units can later be recombined exactly (needed for test T1).

| Factor | What it represents | Method A (advisor, as specified) | Method B (corrected) | Source |
|---|---|---|---|---|
| **C** commercial establishments | business activity | number of places | places ÷ land km² | Overture Places 2026-08-19.0, taxonomy top level shopping, food_and_drink, services_and_business, lifestyle_services, lodging; no filter (same rule in both cities) |
| **R** residential establishments | housing | number of dwellings | dwellings ÷ land km² | SP: IBGE CNEFE 2022 private dwellings. Chicago: Census 2020 housing units, blocks cut by a boundary split by area share |
| **H** transportation hubs | public transport | stations + bus stops | (stations + bus stops) ÷ land km² | CTA and SPTrans GTFS. Stations: CTA `L` parent stations, SP metro stop records grouped by name. CPTM excluded (no Metra in the data) |
| **V** building height | vertical form | mean Overture `height` of buildings with a height | GHSL built volume ÷ built surface | Overture: LiDAR-based in Chicago, OSM tags in SP. GHSL: one satellite method in both cities |
| **N** number of streets | network extent | number of road segments | segments ÷ land km² | Overture road segments, ten street classes, each assigned whole by its midpoint |
| **L** average street length | network grain | mean segment length | same as A | as N |

Why these sources: the [source audit](results/SP_CHI/simple_index_source_audit_2026_10_01/README.md) and the [places coverage test](results/SP_CHI/places_cnefe_coverage_2026_10_01/README.md), where Overture matched 20.10% of São Paulo's census-recorded establishment addresses by name and OpenStreetMap 2.35%.

The next cell builds all components and runs the construction checks; a failed check raises an error and stops the notebook.

In [ ]:
comp = ab.build_components()
named(comp).head()

### Construction checks

Each check compares a quantity built here with an independent count of the same thing:

- commercial counts by two different point-in-polygon routes, and against the earlier source audit;
- São Paulo dwellings against the CNEFE total; Chicago housing units allocated plus the remainder outside the city against the block total, and the same block weights reproducing the published population allocation;
- buildings with a height assigned to units against the audit totals;
- GHSL height recombined from volume and surface against the stored values;
- 173 units, no missing value.

In [ ]:
pd.DataFrame(ab.checks).T

In [ ]:
comp.groupby('city')[['land_km2', 'C_count', 'R_count', 'H_stations', 'H_bus', 'V_A_n', 'N_count']].sum().round(1)

## 3. Scaling and the index

Both methods use the advisor's scaling: each factor is **min–max scaled over the 173 units pooled**,

$$x' = \frac{x - \min x}{\max x - \min x} \in [0, 1],$$

and the **index is the sum of the six scaled factors** (equal weights, range 0–6). Built environment (C, R, H, V) is therefore 4/6 of the index and network (N, L) 2/6.

The only differences between A and B are the ones in the factor table: counts per unit (A) versus counts per km² of land (B), and the height source.

In [ ]:
out, res, S1 = ab.build_indices(comp)
res = named(res)
cols = ['name', 'city', 'land_km2', 'A_index', 'A_rank', 'B_index', 'B_rank']
print('Top 10, method A'); display(res.sort_values('A_rank')[cols].head(10).round(3))
print('Top 10, method B'); display(res.sort_values('B_rank')[cols].head(10).round(3))

### Worked example: Brás and Grajaú

Grajaú is a large peripheral district; Brás is a small dense central one. Compare their raw factors under both methods.

In [ ]:
raw_cols = [c for c in res.columns if c.startswith(('A_raw_', 'B_raw_'))]
ex = res.loc[['SP:10', 'SP:30'], ['name', 'land_km2'] + raw_cols + ['A_rank', 'B_rank']].T
ex.loc['land ratio Grajaú/Brás'] = [None, res.land_km2['SP:30'] / res.land_km2['SP:10']]
ex

## 4. Tests of method bias (protocol P-AB-1)

"Bias" is defined operationally: **a measure of urban intensity should not change merely because unit boundaries are drawn differently, and should not rise mechanically with unit size.** The tests and their criteria were fixed before any result:

| Test | Question | Criterion fixed in advance |
|---|---|---|
| **T1** boundary invariance (decisive) | When neighbouring districts are reported as one unit, is the merged value within the range of its parts? | boundary-invariant if true for ≥ 95% of merged units |
| **T2** size dependence | Does the factor or index correlate with land area? | size-dependent if the 95% bootstrap interval of Spearman's ρ excludes 0 and \|ρ\| ≥ 0.3 |
| **T3** agreement | Do A and B rank the units the same way? | substantively disagree if Spearman < 0.8 or top-10 overlap < 7 |
| **T4** city composition | How many Chicago units are in the top and bottom 20? | descriptive, against Chicago's share of units (77/173) |
| **S1** sensitivity | Does merging paired bus stops (same name within 50 m) change the result? | descriptive |

### T1 — boundary invariance

**Intuition.** Reporting two neighbouring districts as one area changes nothing on the ground. An intensity measure of the merged area should therefore sit between the values of its parts: it cannot be more intense than its most intense part.

**Procedure.** The 96 SP districts are merged into their subprefeituras (`analysis/config/sp_district_subprefeitura.csv`); 29 of the 32 contain two or more districts. Every factor is recomputed for the merged units by the same definition (sum counts and land, recombine means from their sums). The index of a merged unit uses the 173-unit min–max unchanged.

**Why the outcome is a mathematical certainty, and the data measures its size.**

- *Count* (method A): $x_{\text{merged}} = \sum_i x_i$, which exceeds $\max_i x_i$ whenever two or more parts are positive.
- *Ratio of sums* (method B, and the means V and L): $\frac{\sum_i a_i}{\sum_i b_i}$ is a weighted average of the parts' ratios $\frac{a_i}{b_i}$ with weights $\frac{b_i}{\sum b}$, so it always lies between the smallest and largest part.

Column `merged_over_max_part_median` shows how many times the merged value exceeds the largest part.

In [ ]:
t1, t1_detail = ab.test_t1(comp, out)
t1

Merged units whose value fell outside the range of their parts, method B. A sum of separately scaled factors can fall just outside the range of the parts' sums even when every factor is within range:

In [ ]:
t1_detail[(t1_detail.method == 'B') & ~t1_detail.within]

### T2 — size dependence

**Intuition.** If a factor tracks unit size, the index partly ranks units by area. T2 measures the Spearman rank correlation of each factor and of the index with land area, pooled and within each city, with a 95% bootstrap interval (2,000 resamples, seed 20261001).

**How to read it together with T1.** Real urban structure can also correlate with size (large districts are often peripheral and less dense), so T2 alone cannot prove a method biased. A positive correlation in A combined with the T1 failure shows a mechanical size effect; a negative correlation in B is consistent with less dense peripheries, but T2 alone cannot prove it reflects urban form.

In [ ]:
t2 = ab.test_t2(comp, out)
t2.pivot_table(index=['factor'], columns=['method', 'scope'], values='spearman').round(3)

In [ ]:
t2[t2.factor == 'index']

### T3 — agreement between A and B

If both methods measured the same thing, they would rank the units similarly. Brás is the project's reference district.

In [ ]:
t3 = ab.test_t3(res, comp)
t3['unit_with_max_shift_name'] = NAMES[t3['unit_with_max_shift']]
pd.Series(t3)

### T4 — city composition

Chicago has 77 of the 173 units, so about 8.9 Chicago units would appear in a top or bottom 20 if city did not matter. Neither method can tell from available data whether a city imbalance reflects real differences or unequal sources (Overture Places provenance differs by city; dwellings come from two different censuses).

In [ ]:
ab.test_t4(res, comp)

### S1 — merged bus stops

CTA lists most bus stops as a pair, one per side of the street; SPTrans less often. Merging stops with the same normalized name within 50 m tests whether that convention drives the result.

In [ ]:
ab.test_s1(out, S1)

## 5. Application: the Chicago area most similar to Brás

The project's goal is to use the model to choose a Chicago Community Area similar to Brás. With the agreed one-number index, **the most similar area is the one whose index value is closest to Brás's**: $\text{gap}(u) = |I(u) - I(\text{Brás})|$, smallest first. Each method chooses on its own index.

Diagnostic (not part of the agreed model): `profile_distance` is the Euclidean distance between the six scaled factors of the area and of Brás. A large profile distance means the index matched a total built from a different mix of factors.

In [ ]:
match = ab.match_chicago(res)
for m in ('A', 'B'):
    print('Method', m); display(match[match.method == m].head(5))

The areas whose six-factor profile is closest to Brás, with their position in the index ranking:

In [ ]:
match.sort_values(['method', 'rank_by_profile_distance']).groupby('method').head(3)

In [ ]:
F = ab.FACTORS
picks = {'A': match[(match.method == 'A') & (match.rank_by_index_gap == 1)].unit_id.iloc[0],
         'B': match[(match.method == 'B') & (match.rank_by_index_gap == 1)].unit_id.iloc[0]}
prof = pd.DataFrame({f'{m} scaled: {lab}': [res.at[u, f'{m}_scaled_{f}'] for f in F]
                     for m in ('A', 'B') for lab, u in (('Brás', 'SP:10'), (NAMES[picks[m]], picks[m]))}, index=F)
prof.round(3)

## 6. Reproducibility check against the published report

The tables below were written by `build_simple_index_ab.py` and summarized in the methodology report. This cell asserts that the notebook reproduces them exactly.

In [ ]:
PUB = ROOT / 'analysis/results/SP_CHI/simple_index_ab_2026_10_01/tables'
pd.testing.assert_frame_equal(t1.reset_index(drop=True), pd.read_csv(PUB / 't1_boundary_invariance.csv'), check_dtype=False)
pd.testing.assert_frame_equal(t2.reset_index(drop=True), pd.read_csv(PUB / 't2_size_dependence.csv'), check_dtype=False)
pub_idx = pd.read_csv(PUB / 'index_A_B.csv').set_index('unit_id')
for c in ['A_index', 'B_index', 'A_rank', 'B_rank']:
    pd.testing.assert_series_equal(res[c], pub_idx[c], check_dtype=False, check_names=False)
pd.testing.assert_frame_equal(match.reset_index(drop=True), pd.read_csv(PUB / 'bras_chicago_match.csv'), check_dtype=False)
pub_t3 = pd.read_csv(PUB / 't3_agreement.csv').iloc[0]
assert all(str(pub_t3[k]) == str(v) for k, v in t3.items() if k in pub_t3), 'T3 differs'
print('Notebook reproduces the published tables exactly.')

## 7. How to interpret the results

- **T1 is the decisive test.** A factor that fails it changes value when boundaries are redrawn, so it cannot be read as urban intensity. Under the pre-registered criterion, method A's count factors (C, R, H, N) and its index fail and method B passes.
- **T2 and T3 show the practical consequence:** whether A's index rises with unit size, and how differently A and B rank the same places, including Brás.
- **Application:** each method picks the Chicago area with the closest index value to Brás. The profile distance shows whether that area also resembles Brás factor by factor; a one-number index can match totals built from different mixes, which is the information loss accepted for this first sketch.
- **What B does not fix:** both methods use the same sources, so uneven Overture coverage across districts (3.2%–35.2% of CNEFE establishment addresses, after subtracting chance matches) affects both, and the cross-city comparison (T4) cannot be validated with any available reference.
- **Not changed in B, by agreement:** average street length enters positively although longer segments usually mean a coarser network; min–max scaling lets single extreme units set each scale.